# 01 - Data Extraction & Feature Engineering (Module-First)

Notebook nay chi dieu phoi ETL thong qua wrapper module `run_notebook01_etl`.

Luong chay:
1. Cau hinh date range, corridor/segment ids, output path
2. Goi `run_notebook01_etl(config)`
3. Kiem tra schema report + class distribution
4. Preview parquet dau ra

In [ ]:
import os
from pathlib import Path

import pandas as pd

from src.pipelines.notebook01_etl import Notebook01ETLConfig, run_notebook01_etl

In [ ]:
def _parse_ids(raw_value: str) -> list[int]:
    if not raw_value.strip():
        return []
    return [int(x.strip()) for x in raw_value.split(',') if x.strip()]

START_DATE = os.getenv('ETL_START_DATE', '2026-03-25')
END_DATE = os.getenv('ETL_END_DATE', '2026-04-25')
OUTPUT_PATH = os.getenv('ETL_OUTPUT_PATH', '/workspace/ai-core/notebooks/01_processed_features.parquet')

# Vi du: ETL_CORRIDOR_IDS="136550177913819656,392537437542429252"
# Hoac ETL_SEGMENT_IDS="123,456"
CORRIDOR_IDS = _parse_ids(os.getenv('ETL_CORRIDOR_IDS', ''))
SEGMENT_IDS = _parse_ids(os.getenv('ETL_SEGMENT_IDS', ''))
PEAK_HOURS_ONLY = os.getenv('ETL_PEAK_HOURS_ONLY', 'true').lower() == 'true'

config = Notebook01ETLConfig(
    start_date=START_DATE,
    end_date=END_DATE,
    output_path=OUTPUT_PATH,
    corridor_ids=CORRIDOR_IDS or None,
    segment_ids=SEGMENT_IDS or None,
    peak_hours_only=PEAK_HOURS_ONLY,
)

print('Date range :', START_DATE, '->', END_DATE)
print('Output path:', OUTPUT_PATH)
print('Corridors  :', CORRIDOR_IDS if CORRIDOR_IDS else '(none)')
print('Segments   :', SEGMENT_IDS if SEGMENT_IDS else '(none)')
print('Peak hours :', PEAK_HOURS_ONLY)

Date range: 2026-03-25 -> 2026-04-25


In [ ]:
RUN_ETL = True

if RUN_ETL:
    result = run_notebook01_etl(config)
    print('Saved       :', result.output_path)
    print('Shape       :', (result.rows, result.columns))
    print('Class counts:', result.class_counts)
    print('Missing cols:', result.schema_report.get('missing_columns'))
else:
    print('Dry-run: set RUN_ETL=True to execute ETL wrapper.')

Using DB URL (masked): postgresql://psql-smart-traffic-dev.postgres.database.azure.com:5432/traffic_ioc_db  schema= public


In [ ]:
output_path = Path(config.output_path)
assert output_path.exists(), f'Missing output parquet: {output_path}'

df = pd.read_parquet(output_path)
print('Reloaded shape:', df.shape)
print('Columns:', len(df.columns))
print('Class counts:\n', df['congestion_level'].value_counts().sort_index())
df.head(5)

Extracted rows: 3895027


,segment_key,timestamp,current_speed_kmh,traffic_index,delay_seconds,quality_flag,congestion_level,is_closed,free_flow_speed_kmh,default_lane_count,ward_district_id,tomtom_frc,is_one_way
0,1148085288473428275,2026-03-25 06:08:48.740356,45.0,0.0,0,9,0,False,None,None,None,None,True
1,60790401928534190,2026-03-25 06:08:48.740356,33.0,0.0,0,9,0,False,None,None,None,None,True
2,33430329331132128,2026-03-25 06:08:48.740356,45.0,0.0,0,9,0,False,None,None,None,None,True


In [ ]:
# Optional quick diagnostics
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print('Top missing columns (%):')
print(missing_pct.head(15))

tomtom_frc             100.0
segment_key              0.0
timestamp                0.0
current_speed_kmh        0.0
traffic_index            0.0
delay_seconds            0.0
quality_flag             0.0
congestion_level         0.0
is_closed                0.0
free_flow_speed_kmh      0.0
default_lane_count       0.0
ward_district_id         0.0
is_one_way               0.0
dtype: float64

In [ ]:
# Placeholder: feature-level deep dives (if needed) should call module helpers, not inline ETL logic.

In [ ]:
# Placeholder: add optional visualization/reporting cells here.